# DX799 Week 8: K-Nearest Neighbors and Distance Metrics
**Integrated Capstone Project:** VFX, Animation, and the Film Industry  
**Research question:** Can K-Nearest Neighbors predict whether a movie earns at least twice its reported production budget, and how much does the distance metric affect performance?

In my earlier notebooks, I tried to predict movie revenue using regression, and then switched to a classification approach in Weeks 4 through 6. This week, I am sticking with that same classification question so that I can make a fair comparison. I will test K-Nearest Neighbors (KNN), examine different values of K, compare Manhattan and Euclidean distances, and see whether the model can improve on the results from my earlier work.

## 1. KNN and my evaluation plan
KNN is different from the models I have tried so far. Instead of learning a regression equation or splitting the data into branches, it looks at the movies that are closest to a new movie based on their features. The model then takes a vote among those neighboring movies to make its prediction.

The number of neighbors, **K**, is important. A very small K may focus too heavily on individual movies, while a large K can smooth over real differences between films. The way distance is measured also matters. **Euclidean distance** uses straight-line distance between feature vectors, while **Manhattan distance** adds up the absolute differences across individual features. We will compare both approaches.

I will keep the same 80/20 split and random seed used in Weeks 4 through 6. I will tune K, distance metric, and vote weighting using **five-fold stratified cross-validation** on the training data. I will reserve the test set for the final comparison and focus on ROC-AUC, accuracy, precision, recall, and F1. Scaling is especially important for KNN because unscaled budget and release year values could dominate the distances.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
                             RocCurveDisplay, PrecisionRecallDisplay, balanced_accuracy_score)
SEED = 42
path = Path('movies.csv')
if not path.exists(): path = Path('/mnt/data/movies.csv')
raw = pd.read_csv(path)
print('Input file:', path)
print('Original data:', raw.shape)

Input file: movies.csv
Original data: (18905, 23)


## 2. Prepare the same modeling sample
I am using the same restrictions as Week 4: unique movie IDs, recorded positive budgets and revenues, release years from 1980 through 2024, valid release dates, and runtimes of 40 to 240 minutes. Financial success is defined as **worldwide revenue at least two times the reported production budget**. This is a simple measure of box-office performance, not actual profitability because it does not include marketing costs or the share of ticket sales retained by theaters.

To avoid using information that would only become available after a movie is released, I am only including budget, runtime, release year, release month, and selected genres. I am not using actual revenue, ratings, post-release popularity, or vote counts to make predictions. Revenue is used only to create the target variable.

In [2]:
df = raw.drop_duplicates(subset=['movie_id']).copy()
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')
for col in ['budget','revenue','runtime']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.loc[(df.budget > 0) & (df.revenue > 0) & df.release_year.between(1980,2024)
            & df.runtime.between(40,240) & df.release_date.notna()].copy()
df['budget_m'] = df.budget / 1e6
df['revenue_m'] = df.revenue / 1e6
df['gross_to_budget'] = df.revenue / df.budget
df['success'] = (df.gross_to_budget >= 2).astype(int)
genres = df.genres.fillna('').str.split('|')
df['animation'] = genres.map(lambda g: int('Animation' in g))
df['release_month'] = df.release_date.dt.month
for genre in ['Action','Comedy','Drama','Horror']:
    df['genre_' + genre.lower()] = genres.map(lambda g: int(genre in g))
for month in range(2,13):
    df[f'month_{month}'] = (df.release_month == month).astype(int)
features = ['budget_m','runtime','release_year','animation','genre_action','genre_comedy',
            'genre_drama','genre_horror'] + [f'month_{i}' for i in range(2,13)]
tr, te = train_test_split(np.arange(len(df)), test_size=.2, random_state=SEED)
train, test = df.iloc[tr].copy(), df.iloc[te].copy()
X_train, X_test = train[features], test[features]
y_train, y_test = train.success, test.success
print(f'Filtered movies: {len(df):,}, training: {len(train):,}, test: {len(test):,}')
print(f'Predictors: {len(features)}; animated films: {df.animation.sum():,}')
print(f'Success share: {df.success.mean():.1%} overall, {y_train.mean():.1%} training, {y_test.mean():.1%} testing')
print('Missing predictor values:', int(X_train.isna().sum().sum() + X_test.isna().sum().sum()))

Filtered movies: 7,079, training: 5,663, test: 1,416
Predictors: 19; animated films: 376
Success share: 43.6% overall, 43.7% training, 43.1% testing
Missing predictor values: 0


## 3. Exploratory analysis: Why distance may be difficult
In this dataset, movie budgets are measured in millions of dollars, runtimes are measured in minutes, and the genre and release-month variables are represented by zeros and ones. If I calculate distances without scaling these features, budget, runtime, and release year could contribute much more to the distance simply because of their numerical scales. That is why I will standardize features in the modeling pipeline.

Another challenge is that two movies with similar budgets and genres can have very different box-office results. KNN can only use the patterns that actually appear in these available features, so I do not expect the model to make perfect predictions.

In [3]:
fig, ax = plt.subplots(1,2, figsize=(11,4.2))
counts = df.success.value_counts().reindex([0,1])
ax[0].bar(['Below 2x budget','At least 2x budget'], counts.values)
ax[0].set(ylabel='Movies', title='Financial success class balance')
ax[1].hist([df.loc[df.success==0,'budget_m'].clip(upper=250),
             df.loc[df.success==1,'budget_m'].clip(upper=250)], bins=30,
            label=['Below 2x budget','At least 2x budget'], alpha=.7)
ax[1].set(xlabel='Production budget ($ millions, clipped at $250M for display)',
          ylabel='Movies', title='Budget distribution by class')
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. How K and the distance metric affect the model
I will test **K = 3, 7, 15, 31, and 61**, both uniform and distance-weighted voting, and two distance measures. Distance weighting gives closer movies more influence over the prediction. I am using the exact same five training folds for every combination, selecting the configuration with the highest mean ROC-AUC. The scaling step is inside the pipeline, which means each validation fold is transformed using only its corresponding training fold.

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
knn_pipe = Pipeline([('scale',StandardScaler()), ('knn',KNeighborsClassifier(algorithm='auto', n_jobs=2))])
grid = {
    'knn__n_neighbors':[3,7,15,31,61],
    'knn__metric':['euclidean','manhattan'],
    'knn__weights':['uniform','distance']
}
search = GridSearchCV(knn_pipe, grid, scoring='roc_auc', cv=cv,
                      n_jobs=2, return_train_score=True, refit=True)
search.fit(X_train,y_train)
cv_table = pd.DataFrame(search.cv_results_)
print('Best training CV ROC-AUC:', round(search.best_score_,4))
print('Selected parameters:', search.best_params_)
print('Top five configurations by cross-validation ROC-AUC:')
print(cv_table[['param_knn__n_neighbors','param_knn__metric','param_knn__weights',
                'mean_train_score','mean_test_score','std_test_score']]
      .sort_values('mean_test_score',ascending=False).head(5).round(4).to_string(index=False))

Best training CV ROC-AUC: 0.612
Selected parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 61, 'knn__weights': 'distance'}
Top five configurations by cross-validation ROC-AUC:
 param_knn__n_neighbors param_knn__metric param_knn__weights  mean_train_score  mean_test_score  std_test_score
                     61         manhattan           distance            1.0000           0.6120          0.0117
                     31         manhattan           distance            1.0000           0.6116          0.0081
                     31         manhattan            uniform            0.6732           0.6089          0.0089
                     61         euclidean           distance            1.0000           0.6053          0.0116
                     31         euclidean           distance            1.0000           0.6040          0.0079


### Interpreting the cross-validation results: Does the winning model generalize?

The cross-validation search compared **20 configurations** (5 values of K × 2 distance metrics × 2 weighting methods). The highest validation ROC-AUC was **0.6120**, using 61 neighbors, Manhattan distance, and distance weighting. However, the second-place model was almost identical: 31 neighbors with Manhattan distance and distance weighting scored **0.6116**. The difference of **0.0004 ROC-AUC** is far smaller than the reported fold-to-fold variation (standard deviations 0.0117 and 0.0081). Therefore, I cannot conclude that K=61 is meaningfully better than K=31.

The training ROC-AUC for the distance-weighted candidates was **1.0000**. This requires special attention: when each training movie predicts its own class, that movie is its own nearest neighbor with distance zero, so distance weighting can effectively memorize its training label. The perfect training score is **not evidence of a perfect generalizable model**. The validation performance of approximately 0.61 is more informative. The comparison with uniform weighting below helps test whether distance weighting is actually necessary.

The cross-validation scores were used for model selection. The reserved test set should be used for final evaluation, **not repeatedly to choose K, metric, or weighting**.

In [5]:
# Training-versus-validation evidence for the most competitive configurations
selected_cols = ['param_knn__n_neighbors','param_knn__metric','param_knn__weights',
                 'mean_train_score','mean_test_score','std_test_score']
ranking = (cv_table[selected_cols].copy()
           .sort_values('mean_test_score',ascending=False).head(10))
ranking['train_minus_cv_auc'] = ranking['mean_train_score'] - ranking['mean_test_score']
display(ranking.round(4))

# Compare distance and uniform weighting using validation scores only.
cv_weight_comparison = (cv_table.groupby(['param_knn__metric','param_knn__weights'])
                        ['mean_test_score'].agg(['max','mean']).round(4))
display(cv_weight_comparison)

,param_knn__n_neighbors,param_knn__metric,param_knn__weights,mean_train_score,mean_test_score,std_test_score,train_minus_cv_auc
19,61,manhattan,distance,1.0000,0.6120,0.0117,0.3880
17,31,manhattan,distance,1.0000,0.6116,0.0081,0.3884
16,31,manhattan,uniform,0.6732,0.6089,0.0089,0.0643
9,61,euclidean,distance,1.0000,0.6053,0.0116,0.3947
7,31,euclidean,distance,1.0000,0.6040,0.0079,0.3960
18,61,manhattan,uniform,0.6439,0.6040,0.0114,0.0399
6,31,euclidean,uniform,0.6651,0.6009,0.0108,0.0641
15,15,manhattan,distance,1.0000,0.6008,0.0014,0.3992
14,15,manhattan,uniform,0.7021,0.5996,0.0059,0.1026
8,61,euclidean,uniform,0.6390,0.5955,0.0115,0.0435


max    mean
param_knn__metric param_knn__weights                
euclidean         distance            0.6053  0.5900
                  uniform             0.6009  0.5871
manhattan         distance            0.6120  0.5978
                  uniform             0.6089  0.5956

## 5. Visualizing KNN tuning
This graph shows how the validation ROC-AUC changes as K increases, separately for Euclidean and Manhattan distance. It also helps answer whether the smallest neighborhood is too sensitive to individual observations and whether larger neighborhoods wash out useful patterns.

In [6]:
fig, axs=plt.subplots(1,2,figsize=(12,4.3),sharey=True)
for ax,metric in zip(axs,['euclidean','manhattan']):
    for weights in ['uniform','distance']:
        sub = cv_table[(cv_table.param_knn__metric == metric) &
                       (cv_table.param_knn__weights == weights)].sort_values('param_knn__n_neighbors')
        ax.plot(sub.param_knn__n_neighbors.astype(int),sub.mean_test_score,marker='o',label=weights)
    ax.set(xlabel='Number of neighbors (K)',title=f'{metric.title()} distance',ylabel='Mean CV ROC-AUC')
    ax.grid(alpha=.2);ax.legend()
fig.suptitle('How K, distance metric, and voting change validation performance')
plt.tight_layout();plt.show()

/tmp/ipykernel_1076/2073429251.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.show()


## 6. Compare scaled and unscaled KNN
To see why scaling is important rather than just assuming it is, I will compare KNN with and without StandardScaler using the **same selected K, distance metric, and vote weighting**. This comparison shows the influence of the feature units. Importantly, the unscaled model is a diagnostic comparison, not another opportunity to tune on the test set.

In [7]:
p = search.best_params_
kwargs = dict(n_neighbors=p['knn__n_neighbors'],metric=p['knn__metric'],weights=p['knn__weights'],n_jobs=2)
scaled_knn=search.best_estimator_
unscaled_knn=KNeighborsClassifier(**kwargs).fit(X_train,y_train)
def evaluate(label,model):
    pred=model.predict(X_test)
    prob=model.predict_proba(X_test)[:,1]
    return {'Model':label,'Accuracy':accuracy_score(y_test,pred),
            'Precision':precision_score(y_test,pred,zero_division=0),
            'Recall':recall_score(y_test,pred,zero_division=0),
            'F1':f1_score(y_test,pred,zero_division=0),
            'Balanced accuracy':balanced_accuracy_score(y_test,pred),
            'ROC-AUC':roc_auc_score(y_test,prob),
            'Average precision':average_precision_score(y_test,prob)}
rows=[evaluate('Scaled KNN (CV selected)',scaled_knn),evaluate('Unscaled KNN (same K)',unscaled_knn)]
comparison=pd.DataFrame(rows).set_index('Model')
print(comparison.round(4).to_string())

                          Accuracy  Precision  Recall      F1  Balanced accuracy  ROC-AUC  Average precision
Model                                                                                                       
Scaled KNN (CV selected)    0.6017     0.5633  0.3421  0.4257             0.5704   0.6153             0.5505
Unscaled KNN (same K)       0.5975     0.5581  0.3224  0.4087             0.5643   0.6016             0.5416


## 7. Compare KNN with previous classification approaches
Week 4 and Week 5 used logistic regression and SVMs, while Week 6 explored decision trees and random forests. To make comparisons as fair as possible, I will re-fit the same **scaled logistic regression baseline** from Week 4 and use the same test subset. I will also report the earlier Week 6 random forest ROC-AUC as historical context. That earlier result came from a separate notebook, so a full head-to-head comparison would require retraining the random forest under the exact same protocol.

In [8]:
log_grid = GridSearchCV(Pipeline([('scale',StandardScaler()),
                                  ('clf',LogisticRegression(max_iter=6000,solver='liblinear',random_state=SEED))]),
                        {'clf__C':[.001,.01,.1,1,10,100]},scoring='roc_auc',cv=cv,n_jobs=2)
log_grid.fit(X_train,y_train)
log_model=log_grid.best_estimator_
comparison = pd.concat([comparison,pd.DataFrame([evaluate('Week 4 scaled logistic (refit)',log_model)]).set_index('Model')])
print('Logistic baseline selected C:',log_grid.best_params_)
print(comparison.round(4).to_string())
print('Historical Week 6 random forest test ROC-AUC reported in earlier notebook: approximately 0.651')

Logistic baseline selected C: {'clf__C': 0.01}
                                Accuracy  Precision  Recall      F1  Balanced accuracy  ROC-AUC  Average precision
Model                                                                                                             
Scaled KNN (CV selected)          0.6017     0.5633  0.3421  0.4257             0.5704   0.6153             0.5505
Unscaled KNN (same K)             0.5975     0.5581  0.3224  0.4087             0.5643   0.6016             0.5416
Week 4 scaled logistic (refit)    0.5911     0.5432  0.3290  0.4098             0.5595   0.6137             0.5376
Historical Week 6 random forest test ROC-AUC reported in earlier notebook: approximately 0.651


## 8. ROC curves, precision-recall curves, and confusion matrix
ROC-AUC measures how effectively a model ranks successful films above unsuccessful ones across different thresholds. A confusion matrix helps show what errors the model makes when it uses the default classification threshold. Precision-recall curves are helpful because accuracy alone can hide the fact that a model is missing successful movies.

In [9]:
fig,axs=plt.subplots(1,2,figsize=(12,4.6))
for label,model in [('Scaled KNN',scaled_knn),('Unscaled KNN',unscaled_knn),('Scaled logistic',log_model)]:
    RocCurveDisplay.from_estimator(model,X_test,y_test,ax=axs[0],name=label)
    PrecisionRecallDisplay.from_estimator(model,X_test,y_test,ax=axs[1],name=label)
axs[0].set_title('Test ROC curves'); axs[1].set_title('Test precision-recall curves')
plt.tight_layout();plt.show()
fig,ax=plt.subplots(figsize=(5,4.3))
ConfusionMatrixDisplay.from_predictions(y_test,scaled_knn.predict(X_test),ax=ax,
                                        display_labels=['Below 2x','At least 2x'],cmap='Blues',colorbar=False)
ax.set_title('Scaled KNN: Test confusion matrix')
plt.tight_layout();plt.show()

/tmp/ipykernel_1076/3734553246.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.show()
/tmp/ipykernel_1076/3734553246.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.show()


## 9. Check overfitting
KNN has no coefficient regularization like Lasso or Ridge, but the choice of K controls how closely it fits the data. A small K can effectively memorize individual training movies. A larger K usually creates a smoother boundary. I will compare training, cross-validation, and held-out ROC-AUC for the selected model. This is especially relevant for distance-weighted KNN, which can classify training examples almost perfectly because each movie is its own nearest neighbor.

### What the overfitting measures mean

A training-to-test ROC-AUC difference must be interpreted carefully for **distance-weighted KNN**, because self-neighbor predictions can make training ROC-AUC equal to 1 by construction. I will still report the number for transparency, but rely primarily on **five-fold validation ROC-AUC and held-out test ROC-AUC** to assess generalization. If validation and test performance are similar, the selection process is reasonably stable even if the training score looks dramatically stronger.

A larger K makes neighborhood predictions smoother and can reduce sensitivity to isolated movies. It cannot, however, fix unreliable or uninformative predictors. Even the chosen model may struggle to distinguish commercially successful movies because budget, release timing, and genre omit marketing, audience reception, and other major drivers of revenue.

In [10]:
train_probs=scaled_knn.predict_proba(X_train)[:,1]
test_probs=scaled_knn.predict_proba(X_test)[:,1]
print('Selected KNN training ROC-AUC:',round(roc_auc_score(y_train,train_probs),4))
print('Selected KNN CV ROC-AUC:',round(search.best_score_,4))
print('Selected KNN test ROC-AUC:',round(roc_auc_score(y_test,test_probs),4))
print('Training-to-test ROC-AUC gap:',round(roc_auc_score(y_train,train_probs)-roc_auc_score(y_test,test_probs),4))
print('Selected K:',p['knn__n_neighbors'],'Metric:',p['knn__metric'],'Weights:',p['knn__weights'])

Selected KNN training ROC-AUC: 1.0
Selected KNN CV ROC-AUC: 0.612
Selected KNN test ROC-AUC: 0.6153
Training-to-test ROC-AUC gap: 0.3847
Selected K: 61 Metric: manhattan Weights: distance


In [11]:
# Additional guardrail: compare the best CV-selected distance-weighted model
# with a uniform-weighted counterpart using K and metric fixed BEFORE looking at test data.
uniform_peer = Pipeline([('scale', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=p['knn__n_neighbors'],
                                  metric=p['knn__metric'], weights='uniform', n_jobs=2))])
uniform_peer.fit(X_train, y_train)
for label, model in [('CV-selected distance-weighted KNN',scaled_knn),
                     ('Same K and metric, uniform weighting',uniform_peer)]:
    tr_auc = roc_auc_score(y_train,model.predict_proba(X_train)[:,1])
    te_auc = roc_auc_score(y_test,model.predict_proba(X_test)[:,1])
    print(f'{label}: train ROC-AUC={tr_auc:.4f}; test ROC-AUC={te_auc:.4f}')
# This is a diagnostic comparison, NOT a second round of test-based model selection.

CV-selected distance-weighted KNN: train ROC-AUC=1.0000; test ROC-AUC=0.6153


Same K and metric, uniform weighting: train ROC-AUC=0.6487; test ROC-AUC=0.6148


## Week 8 conclusions for Milestone Two: Specific findings and limitations

**Distance and scaling:** Using the same selected K, metric, and weighting, standardized KNN had a held-out ROC-AUC of **0.6153**, compared with **0.6028** without scaling. The difference is **0.0125 ROC-AUC**, although accuracy was essentially unchanged (60.17% scaled versus 60.24% unscaled). This illustrates why accuracy alone is not enough: scaling improved how the model *ranked* movies across possible thresholds, but not its accuracy at the default classification threshold.

**Performance relative to previous approaches:** The scaled KNN's ROC-AUC of **0.6153** was only **0.0016** above the refit logistic model's **0.6137**. KNN had an F1 of **0.4257** versus **0.4098** for logistic regression. These are modest differences; they do not establish that KNN is reliably superior. A previous Week 6 random-forest classification experiment reported ROC-AUC around **0.651**, but that result should only be compared directly if the dataset, target, predictors, and split were identical.

**A major operational limitation:** Recall was just **0.3421**, so the default 0.5 threshold correctly identified only about **34%** of movies that actually doubled their budget. A studio using the model to screen potential successes would miss many of them. Precision was **0.5633**, so a predicted success was correct around 56% of the time in this sample. Different business objectives might justify a different threshold, but it would need to be chosen using validation data rather than the test set.

**Overfitting and model selection:** Distance-weighted training ROC-AUC of **1.0000** largely reflects self-neighbor memorization, while the five-fold CV score of **0.6120** better captures out-of-sample capability. There is minimal separation between the top two K settings in validation, so I should describe K=61 as **the best observed grid-search configuration**, not as a uniquely optimal value. The standard deviation across folds also shows why a 0.0004 difference is not a robust conclusion.

**Limitations:** The analysis excludes films without recorded positive budget and revenue, the 2× threshold is a gross-revenue proxy rather than profitability, film revenues are not inflation-adjusted, and input features are limited to values plausibly known pre-release. The results describe predictive relationships, not causal effects. They are useful as a baseline but are too weak to support a reliable standalone film-financing decision.

## 10. What the results mean for my Capstone
The goal is not simply to find one model with a slightly higher score. I want to understand whether the method makes sense for the kinds of information I have. KNN is attractive because it compares a new movie with similar historical movies. However, a movie can look similar on paper and still generate a very different amount of revenue. Marketing, franchise strength, critical reception, and competition may all matter, but most of those factors are not included in the current pre-release dataset.

The summary below prints the actual test results so that I can discuss them accurately in Milestone Two rather than making assumptions before running the models.

In [12]:
knn_r=comparison.loc['Scaled KNN (CV selected)']
log_r=comparison.loc['Week 4 scaled logistic (refit)']
print(f'Our Week 8 analysis used {len(df):,} movies and {len(features)} pre-release predictors.')
print(f'The selected KNN used K={p["knn__n_neighbors"]}, {p["knn__metric"]} distance, and {p["knn__weights"]} weighting.')
print(f'KNN holdout ROC-AUC was {knn_r["ROC-AUC"]:.3f}, accuracy was {knn_r["Accuracy"]:.1%}, and F1 was {knn_r["F1"]:.3f}.')
print(f'The logistic comparison reached ROC-AUC {log_r["ROC-AUC"]:.3f} and F1 {log_r["F1"]:.3f}.')
print('The scaled versus unscaled comparison, K tuning plot, and train/test gap should guide the discussion of distance and overfitting.')
print('A limitation is that the data only includes movies with positive reported budgets and revenues, so results may not generalize to all films.')

Our Week 8 analysis used 7,079 movies and 19 pre-release predictors.
The selected KNN used K=61, manhattan distance, and distance weighting.
KNN holdout ROC-AUC was 0.615, accuracy was 60.2%, and F1 was 0.426.
The logistic comparison reached ROC-AUC 0.614 and F1 0.410.
The scaled versus unscaled comparison, K tuning plot, and train/test gap should guide the discussion of distance and overfitting.
A limitation is that the data only includes movies with positive reported budgets and revenues, so results may not generalize to all films.
